# 19 Push-and-Rotate：规则式重排


本课按百科条目写。规则法不维护冲突树，按「挡住就推开」走。调度台在最后。

---

## 1. 定义

**Push-and-Rotate**（de Wilde et al.）及更早 Push-and-Swap：不搜 CT，按规则

- **Push**：沿最短路走，挡路者被推到空邻格
- **Swap / Rotate**：在环上轮转换位

无向、空位数 ≥ 2 时一类图上完备。本课教学子集 **只 push**：空位不够或有向无反边则 `stuck` / `directed`。

---

## 2. 和搜索法

| | CBS | Push |
|--|--|--|
| 对象 | 整条时空路径 | 当前构型上的一步规则 |
| 最优 | SOC 最优 | 否 |
| 有向窄廊 | 能等（若图可解） | 没旁格可推则失败 |

规则法常数小、好做执行层；失败要换搜索。

---

## 3. 不变量

- 每步之后两车不同格。
- 无向网格：邻居可双向走。
- 有向图缺反向边：本实现直接 `found=False`（必看负例）。

---

## 4. 伪代码（只 push）

```
重复直到都到达或卡住:
    for 未到目标的车 a:
        nxt ← A*(pos[a], goal[a]) 的下一步
        if nxt 空: 走
        else: 把占 nxt 的车推到它的空邻居；成功则 a 再走
```

完整论文还有 rotate 检测环。教学版空位不够会 `stuck`。

---

## 5. 手算

4×3 空网格，a 在 (0,0) 去 (0,2)，b 在 (3,0) 去 (3,2)。各走各列，push 用不上，几步到齐。

有向 `L→M→R` 对头：没有「旁边」可推，也不能换边。算法应报告失败，不是 bug。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：逐步 push


In [ ]:
from lib.maps import grid_open, directed_deadlock_map
from lib.search import astar
from lib.pretty import draw_grid
from lib.agents import soc


def push_simple(graph, starts, goals, steps=40, verbose=True):
    pos = dict(starts)
    paths = {a: [starts[a]] for a in starts}
    occupied = {v: k for k, v in pos.items()}
    for t in range(steps):
        if all(pos[a] == goals[a] for a in starts):
            return {"found": True, "paths": paths, "soc": soc(paths)}
        progressed = False
        for a in starts:
            if pos[a] == goals[a]:
                continue
            r = astar(graph, pos[a], goals[a], record=False)
            if not r.found or len(r.path) < 2:
                continue
            nxt = r.path[1]
            if nxt not in occupied:
                occupied.pop(pos[a], None)
                pos[a] = nxt
                occupied[nxt] = a
                paths[a].append(nxt)
                progressed = True
            else:
                other = occupied[nxt]
                for cand, _ in graph.neighbors(nxt):
                    if cand not in occupied:
                        occupied.pop(pos[other], None)
                        pos[other] = cand
                        occupied[cand] = other
                        paths[other].append(cand)
                        occupied.pop(pos[a], None)
                        pos[a] = nxt
                        occupied[nxt] = a
                        paths[a].append(nxt)
                        progressed = True
                        if verbose:
                            print(f"t={t} {a} 推开 {other} -> {cand}，自己到 {nxt}")
                        break
        if verbose:
            print(f"t={t} pos={pos}")
        if not progressed:
            return {"found": False, "paths": paths, "reason": "stuck"}
    return {"found": all(pos[a] == goals[a] for a in starts), "paths": paths}


g = grid_open(4, 3)
print(draw_grid(4, 3, start=(0, 0), goal=(0, 2)))
r = push_simple(g, {"a": (0, 0), "b": (3, 0)}, {"a": (0, 2), "b": (3, 2)})
print("无向 found", r["found"], "SOC", r.get("soc"))


In [ ]:
from lib.algos.rules import push_and_rotate

print("库无向", push_and_rotate(g, {"a": (0, 0), "b": (3, 0)}, {"a": (0, 2), "b": (3, 2)})["found"])
print("库有向", push_and_rotate(directed_deadlock_map(), {"a": "L", "b": "R"}, {"a": "R", "b": "L"}))


## 常见 bug

1. 对向推形成振荡（A 推 B，B 又推回 A）。本实现会 stuck。
2. 有向图当无向 push。
3. 推到别人目标上把别人锁死，没有 rotate 就解不开。

## 条目小结

| 项目 | 内容 |
|------|------|
| 动作 | push（教学）/ rotate（论文） |
| 完备 | 无向 + 足够空位 |
| 有向 | 应失败 |
| 下一课 | 每拍抢格：PIBT |

## 练习

1. 两车对头、中间无空格，push 为何振荡？
2. 空位数=1 的环上，为何需要 rotate 而不是 push？
3. 有向死锁失败是图的问题还是算法的问题？两者都是。


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), r, "Push-and-Rotate")
studio
